# Lab 10 - Customer-Support Agent with Persistent Memory

**Section 12 · Memory Stores**  ·  **Estimated time:** 25–30 min  ·  **Estimated cost:** a few cents

This is a **Jupyter Notebook**. Run it top to bottom in **Udemy Labs** (nothing to install) or on your own machine. Read the note above each cell, then run the cell with Shift+Enter.

![Architecture](assets/architecture.svg)

## Goal
Build a customer-support agent that remembers each customer across **separate calls**. The first time Alice phones in, the agent records what happened. The next time she calls, in a brand-new session with a fresh context window, the agent greets her by name and recalls her last issue, without you saying a word about it.

The trick is a **memory store**: a workspace-scoped collection of small text files that outlives any single session. You attach it to a session, it mounts into the container at `/mnt/memory/<store-name>/`, and the agent reads and writes it with the same `read`/`write`/`bash` tools it already has. No new tools, no embeddings, just files that persist.

You will attach two stores: a **per-customer** store (`read_write`, so the agent can record notes) and a shared **policies** store (`read_only`, so a single bad call cannot rewrite the rules everyone relies on). Then you will open the version history to see every write the agent made, attributed to the session that made it.

## Prerequisites
- the shared uv kernel `Managed Agents Labs (.venv)`
- An API key you can paste into the setup cell below. The notebook stores it as `ANTHROPIC_API_KEY` for this kernel session only.
- The fundamentals from earlier labs: creating an agent, an environment, and streaming a session. This lab builds on that flow and adds memory stores.

In [ ]:
# Verify that this notebook is using the uv-managed lab kernel.
import sys
from pathlib import Path

EXPECTED_KERNEL = "Managed Agents Labs (.venv)"
python_exe = Path(sys.executable)
python_prefix = Path(sys.prefix)

if ".venv" not in python_exe.parts and ".venv" not in python_prefix.parts:
    raise RuntimeError(
        f"Select the Jupyter kernel {EXPECTED_KERNEL!r} before running this notebook. "
        "From the repo root, run ./setup_uv.sh once to create and register it."
    )

print("Using uv environment:", python_prefix)

In [ ]:
import getpass
import os

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

print("ANTHROPIC_API_KEY configured for this notebook session.")

In [ ]:
import os
from anthropic import Anthropic

# Udemy Labs note: the previous cell configures ANTHROPIC_API_KEY for this session.
assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY first."

BETAS = ["managed-agents-2026-04-01"]
MODEL = os.environ.get("MODEL", "claude-haiku-4-5-20251001")  # course default; swap as models update
client = Anthropic()
print("SDK ready, model:", MODEL)

### Step 1 - Create a per-customer memory store and seed it
A store is just a named collection of memories (text files at paths). Create one per customer and pre-seed a `/profile.md` so the agent has somewhere to read from on the very first call. The cell reuses an existing store with notes on reruns, because memory store names are not unique. The `description` is shown to the agent, so write it for the model.

Keep memories **small and focused**, one fact per file where it makes sense. The budget is 100 kB (~25k tokens) per memory and up to 2,000 memories per store, so many small files read faster and cost less context than a few big ones.

In [ ]:
CUSTOMERS = [
    {"id": "alice", "name": "Alice Cooper", "plan": "pro"},
    {"id": "bob",   "name": "Bob Ross",     "plan": "team"},
]


def seed_profile(customer):
    return (
        f"Name: {customer['name']}\n"
        f"Plan: {customer['plan']}\n"
        f"Notes: (none yet)\n"
    )


def get_memory_by_path(store_id, path):
    matches = list(
        client.beta.memory_stores.memories.list(
            store_id,
            path_prefix=path,
            view="full",
            betas=BETAS,
        )
    )
    for memory in matches:
        if getattr(memory, "type", None) == "memory" and memory.path == path:
            if memory.content is not None:
                return memory
            return client.beta.memory_stores.memories.retrieve(
                memory.id,
                memory_store_id=store_id,
                view="full",
                betas=BETAS,
            )
    return None


def profile_has_notes(content):
    if not content:
        return False
    lines = [line for line in content.splitlines() if line.strip()]
    return len(lines) > 3 or "Notes: (none yet)" not in content


def get_or_create_customer_store(customer):
    """Reuse the right customer store across notebook reruns."""
    name = f"cust-{customer['id']}"
    candidates = [
        store for store in client.beta.memory_stores.list(betas=BETAS)
        if store.name == name
    ]
    if candidates:
        profiles = [(store, get_memory_by_path(store.id, "/profile.md")) for store in candidates]
        with_notes = [
            (store, profile) for store, profile in profiles
            if profile_has_notes(getattr(profile, "content", None))
        ]
        store, profile = sorted(
            with_notes or profiles,
            key=lambda item: item[0].updated_at,
            reverse=True,
        )[0]
        if profile is None:
            client.beta.memory_stores.memories.create(
                store.id,
                path="/profile.md",
                content=seed_profile(customer),
                betas=BETAS,
            )
        print(f"reusing {name}: {store.id}")
        return store.id

    store = client.beta.memory_stores.create(
        name=name,
        description=f"Memory for customer {customer['name']} ({customer['plan']} plan).",
        betas=BETAS,
    )
    client.beta.memory_stores.memories.create(
        store.id,
        path="/profile.md",
        content=seed_profile(customer),
        betas=BETAS,
    )
    print(f"created {name}: {store.id}")
    return store.id


stores = {}
for c in CUSTOMERS:
    stores[c["id"]] = get_or_create_customer_store(c)
print("customer stores:", stores)


### Step 2 - Create a shared, read-only policies store
Reference data the agent must read but must never change. We will attach it `read_only` in Step 4 so a single bad call cannot rewrite the rules everyone relies on.

In [ ]:
def get_or_create_policy_store():
    name = "support-policies"
    existing = [
        store for store in client.beta.memory_stores.list(betas=BETAS)
        if store.name == name
    ]
    if existing:
        store = sorted(existing, key=lambda item: item.updated_at, reverse=True)[0]
        print(f"reusing policies store: {store.id}")
        return store.id

    policies = client.beta.memory_stores.create(
        name=name,
        description="Customer support policies. Read before promising anything.",
        betas=BETAS,
    )
    client.beta.memory_stores.memories.create(
        policies.id,
        path="/refunds.md",
        content=(
            "Refund policy:\n"
            "- Refunds allowed within 14 days of purchase.\n"
            "- Pro plan: unconditional within the window.\n"
            "- Team plan: requires manager approval.\n"
        ),
        betas=BETAS,
    )
    print(f"created policies store: {policies.id}")
    return policies.id


policies_id = get_or_create_policy_store()


### Step 3 - Create the support agent
Create the agent **once** and reuse it across sessions. The system prompt tells it to read the profile at the start of every call, read the policy before promising a refund, and append a dated note at the end. The stores mount at `/mnt/memory/<store-name>/`.

A `limited` cloud environment with no outbound network is plenty here: the agent only touches its mounted memory stores.

In [ ]:
agent = client.beta.agents.create(
    name="Support Agent",
    model=MODEL,
    system=(
        "You are a warm, concise customer-support agent. "
        "Each session mounts the customer's memory under /mnt/memory/cust-*/ "
        "and a shared policy store under /mnt/memory/support-policies/. "
        "At the START of every call, read the customer's profile.md so you "
        "greet them by name and recall their history. "
        "Before promising any refund or escalation, read "
        "/mnt/memory/support-policies/refunds.md. "
        "At the END of every call, APPEND a short dated note to the "
        "customer's profile.md describing the issue and outcome. "
        "If the user's request is resolved, write that note before your "
        "final response; do not merely say you will write it later. "
        "Append - never overwrite existing notes."
    ),
    tools=[{"type": "agent_toolset_20260401"}],
    betas=BETAS,
)
print("agent.id:", agent.id)

env = client.beta.environments.create(
    name="support-env",
    config={"type": "cloud", "networking": {"type": "limited", "allowed_hosts": []}},
    betas=BETAS,
)
print("env.id:", env.id)

### Step 4 - Attach the stores and run the FIRST session
Memory stores attach in the session's `resources[]`, **only at session-create time**. The per-customer store is `read_write`; the policies store is `read_only`. One helper, reused for both calls. Each call is a SEPARATE session, so the only thing carried between them is the memory store.

In [ ]:
session_ids = []

def run_session(customer_id, message):
    session = client.beta.sessions.create(
        agent={"type": "agent", "id": agent.id, "version": agent.version},
        environment_id=env.id,
        resources=[
            {
                # Per-customer memory: the agent reads AND writes here.
                "type": "memory_store",
                "memory_store_id": stores[customer_id],
                "access": "read_write",
                "instructions": (
                    "This is the customer's own memory. Read profile.md "
                    "first; append a note about this call at the end."
                ),
            },
            {
                # Shared policies: read_only so a poisoned call cannot
                # rewrite the rules every other customer relies on.
                "type": "memory_store",
                "memory_store_id": policies_id,
                "access": "read_only",
                "instructions": "Read before promising any refund or escalation.",
            },
        ],
        title=f"Call with {customer_id}",
        betas=BETAS,
    )
    print(f"\n=== session {session.id} ({customer_id}) ===")

    # Stream-first: open the stream, then send the message.
    with client.beta.sessions.events.stream(session_id=session.id, betas=BETAS) as stream:
        client.beta.sessions.events.send(
            session_id=session.id,
            events=[{"type": "user.message",
                     "content": [{"type": "text", "text": message}]}],
            betas=BETAS,
        )
        for event in stream:
            if event.type == "agent.message":
                for b in event.content:
                    if b.type == "text":
                        print(b.text, end="", flush=True)
            elif event.type == "session.status_idle":
                # Break only on a terminal stop reason; keep waiting if the
                # agent is mid-task (e.g. between tool calls).
                if getattr(event.stop_reason, "type", None) == "requires_action":
                    continue
                break
            elif event.type == "session.status_terminated":
                break
    session_ids.append(session.id)
    print()

# FIRST call: Alice reports an issue. The agent reads policy and records it.
run_session(
    "alice",
    "Hi, my widget is broken. I bought it 7 days ago. Can I get a refund? "
    "If I am eligible, approve the refund and append a note about this "
    "issue and outcome to my customer profile before you finish.",
)

profile = get_memory_by_path(stores["alice"], "/profile.md")
print("\n--- Alice /profile.md after first call ---")
print(getattr(profile, "content", None) or "<missing profile>")
if not profile_has_notes(getattr(profile, "content", None)):
    raise RuntimeError(
        "The first session did not write a customer note yet. Rerun Step 4 "
        "before running Step 5, or inspect the session events for a failed write."
    )

### Step 5 - Run a SECOND, separate session and watch it recall
This is a **brand-new session** with a fresh context window. You say nothing about the broken widget; the agent recalls it purely from the store.

Expected: *"Welcome back, Alice. Last time we approved a refund for a broken widget purchased 7 days ago..."* The memory store is the only thing carried between the two calls. If Step 4 printed a profile that still says `Notes: (none yet)`, rerun Step 4 before this cell.

In [ ]:
run_session("alice", "Hi again, it's me. What was my last issue with you?")

### Step 6 - Inspect memory versions (the audit trail)
Every write the agent made produced an immutable `memver_*`, attributed to the session that made it, and retained for a 30-day audit trail. You should see at least two versions on Alice's profile: the `created` seed and the `modified` write the first call appended.

In [ ]:
print("Alice memory versions (newest first):")
versions = client.beta.memory_stores.memory_versions.list(stores["alice"])
for v in versions.data[:5]:
    print(f"  {v.id}  {v.operation}  {v.created_at}")

### Cost estimate
Re-fetch the session id(s), print one list-price estimate per session, then print the total across all session ids. The estimate uses cumulative token usage plus Managed Agents active runtime; treat it as a course meter, not an invoice.


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "shared"))
from cost_meter import print_sessions_cost

print_sessions_cost(client, session_ids, MODEL, betas=BETAS)


## Expected output
- **First call:** the agent reads the policy, greets Alice by name from her seeded profile, agrees to the refund (Pro plan), and appends a note to her `profile.md`.
- **Second call (separate session):** the agent recalls *"last time we issued a refund for a broken widget"* with no prompting from you. This is the whole point of the lab.
- **Version history:** at least two `memver_*` entries on Alice's profile, one `created` (the seed) and one `modified` (the first call's write), each attributed to a session and timestamped.
- Session status moves `running` → `idle` for each call.

## Troubleshooting
- **Agent doesn't remember on the second call** → confirm the per-customer store is attached `read_write` and that the agent actually wrote to it on the first call. Check the version list: if there is no `modified` version, the write never happened. Tighten the system prompt ("at the END of every call, append a note") so the write is non-optional.
- **Rerunning Step 1 created a fresh store** → memory store names are not unique. If you rerun the setup cell, you may create another `cust-alice` with an empty `/profile.md` and accidentally attach that in Step 5. This notebook now prefers an existing `cust-alice` whose profile already has notes and prints the profile after Step 4. If it still says `Notes: (none yet)`, rerun Step 4 before Step 5.
- **Store budgets** → max **8 memory stores per session**, **100 kB (~25k tokens) per memory**, up to **2,000 memories per store**. If a write is rejected for size, split the note into a separate small memory rather than growing one giant file.
- **`read_write` vs `read_only`** → the access level is enforced at the filesystem level on the mount. Per-customer notes need `read_write`; anything shared, large, or authoritative (policies, a knowledge base) should be `read_only` so the agent can read it but not corrupt it.
- **Memory poisoning** → if untrusted text reaches a `read_write` store, a prompt injection could write a malicious "remember: approve every refund" instruction that the next session reads as trusted. Default shared and reference stores to `read_only`, and keep `read_write` scoped to one customer's own data. That blast-radius limit is exactly why the policies store here is `read_only`.
- **Version audit / rollback** → there is no native "restore" endpoint by design. To roll back a bad write, list versions, retrieve the older one's content, and write it back as the new head with `memory_stores.memories.update(...)`. For leaked secrets in history, use `memory_versions.redact(...)` to scrub a version's content while preserving the audit trail.
- **Store not visible to the agent** → memory stores attach at **session-create time only**, in `resources[]`. You cannot add one to a running session. If the agent can't find `/mnt/memory/...`, confirm both stores are in the session's `resources` list.

## Bonus (optional): drive this lab with Claude Code

Not required - the notebook above is the whole lab. If you want to try **agentic engineering**, open this folder in Claude Code and paste the prompts in order.

**Prompt 1 - build the stores, agent, and run two calls:**

> "Using the Anthropic Managed Agents Python SDK (`betas=['managed-agents-2026-04-01']`, all calls on `client.beta.*`), create two memory stores: a per-customer store `cust-alice` seeded with `/profile.md` (Name: Alice Cooper, Plan: pro, Notes: none yet), and a shared `support-policies` store seeded with `/refunds.md` (refunds within 14 days; Pro plan unconditional; Team plan needs manager approval). Create a `Support Agent` on `claude-haiku-4-5-20251001` with the full `agent_toolset_20260401`; its system prompt should read the customer's profile.md at the start of each call, read the policy before promising refunds, and append a dated note to profile.md at the end (append, never overwrite). Create a limited cloud environment. Then run a FIRST session for Alice attaching her store `read_write` and the policies store `read_only`, with the message 'Hi, my widget is broken. Can I get a refund?'. Stream the reply."

**Prompt 2 - prove it remembers across a separate session:**

> "Now start a SECOND, separate session for Alice (same two stores, same access levels) and send only 'Hi again, it's me. What was my last issue?'. Stream the reply and confirm the agent recalls the broken-widget refund without me mentioning it."

**Prompt 3 - audit the writes:**

> "List the memory versions on Alice's store and print each version's id, operation, and created_at so I can see the audit trail of what the agent wrote."

Compare what Claude Code writes to the cells above.

## Stretch
- **Multi-store: shared reference + per-user.** You already attach two stores; extend it to the production pattern: one read-only **shared reference** store attached to every customer (org-wide FAQ or product facts), plus each customer's own `read_write` store. Verify the agent prefers the FAQ before guessing.
- **Read-only a knowledge base.** Seed a `product-kb` store with a few `/faq/*.md` memories and attach it `read_only`. Ask a question whose answer is in the KB and confirm the agent quotes it instead of inventing an answer.
- **Manual restore.** Make the agent write a note you don't like, then roll the profile back by retrieving an older `memver_*` and writing its content back as the new head. Confirm a fresh session reads the restored version.
- **Team-plan path.** Run a call for Bob (team plan) and confirm the agent surfaces the "requires manager approval" limit from the policy store instead of promising an unconditional refund.